# Russian Lip Reading Baseline, Semester 4 - Kaggle

Kaggle version of the first PyTorch baseline on the expanded Russian lip-reading dataset.

Goal: classify a short mouth-crop video clip into one of 10 Russian word classes.


## 1. Kaggle Dataset Setup

This notebook expects the dataset to be attached as a Kaggle Dataset.

In Kaggle, click **Add Input** and add the uploaded lip-reading dataset. Kaggle usually exposes it under `/kaggle/input/<dataset-slug>/`.


In [ ]:
from pathlib import Path
import zipfile
import shutil

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

print("Kaggle input exists:", KAGGLE_INPUT.exists())
if KAGGLE_INPUT.exists():
    print("Available inputs:")
    for item in sorted(KAGGLE_INPUT.iterdir()):
        print("-", item)


## 2. Find Dataset

This cell finds `ru_dataset` inside Kaggle inputs. If the dataset is still stored as a zip file, it extracts it into `/kaggle/working/lipreading_data`.


In [ ]:
def find_ru_dataset():
    # Case 1: Kaggle has already extracted the uploaded zip.
    for candidate in KAGGLE_INPUT.rglob("ru_dataset"):
        if (candidate / "mouth_crops_padded").exists():
            return candidate

    # Case 2: Kaggle input contains the zip as a file.
    zip_names = ["kaggle_lipreading_dataset.zip", "colab_lipreading_dataset.zip"]
    zip_matches = []
    for zip_name in zip_names:
        zip_matches.extend(KAGGLE_INPUT.rglob(zip_name))

    if zip_matches:
        zip_path = zip_matches[0]
        extract_dir = KAGGLE_WORKING / "lipreading_data"
        expected = extract_dir / "ru_dataset" / "mouth_crops_padded"

        if not expected.exists():
            if extract_dir.exists():
                shutil.rmtree(extract_dir)
            extract_dir.mkdir(parents=True, exist_ok=True)

            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                for member in zip_ref.infolist():
                    fixed_name = member.filename.replace("\\", "/")
                    target_path = extract_dir / fixed_name

                    if fixed_name.endswith("/"):
                        target_path.mkdir(parents=True, exist_ok=True)
                        continue

                    target_path.parent.mkdir(parents=True, exist_ok=True)
                    with zip_ref.open(member) as source, open(target_path, "wb") as target:
                        target.write(source.read())

        return extract_dir / "ru_dataset"

    raise FileNotFoundError(
        "Could not find ru_dataset. Add kaggle_lipreading_dataset.zip as a Kaggle Dataset input first."
    )


data_root = find_ru_dataset()
print("data_root:", data_root)
print("mouth crops exist:", (data_root / "mouth_crops_padded").exists())


## 3. Imports, Paths, Tables, Vocabulary

`train_df`, `val_df`, and `test_df` are pandas tables loaded from CSV files.

`vocab` is the list of word classes. `word_to_idx` converts a word into a class number.


In [ ]:
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

split_name = "ml_splits_mouth_crops_padded_clean_manual_no_spk09_10"
split_dir = data_root / split_name / "speaker_top10"

if not (split_dir / "train.csv").exists():
    split_name = "ml_splits_mouth_crops_padded_no_spk09_10"
    split_dir = data_root / split_name / "speaker_top10"

print("split_dir:", split_dir)

train_df = pd.read_csv(split_dir / "train.csv")
val_df = pd.read_csv(split_dir / "val.csv")
test_df = pd.read_csv(split_dir / "test.csv")

vocab = (split_dir / "vocab.txt").read_text(encoding="utf-8").splitlines()
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

device = "cuda" if torch.cuda.is_available() else "cpu"

print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))
print("vocab:", vocab)
print("device:", device)
train_df.head()


## 4. Check Class Distribution

This shows that the dataset is imbalanced: some words occur much more often than others.

In [ ]:
print("Train word counts:")
print(train_df["word"].value_counts())

print("\nValidation word counts:")
print(val_df["word"].value_counts())

print("\nTest word counts:")
print(test_df["word"].value_counts())


## 5. Video Loading

`load_video` converts one mp4 clip into a PyTorch tensor.

The default mode is RGB, so the output shape is `[3, frames, height, width]`, for example `[3, 24, 96, 96]`.

For the grayscale experiment we can call `load_video(..., grayscale=True)`. Then the output shape is `[1, frames, height, width]`.



In [ ]:
def load_video(path, num_frames=24, size=96, grayscale=False):
    cap = cv2.VideoCapture(str(path))

    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        raise ValueError(f'Cannot read video: {path}')

    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()

        if not ok:
            continue

        if grayscale:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            frame = cv2.resize(frame, (size, size))
            frame = frame[:, :, None]
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (size, size))

        frame = frame / 255.0
        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        raise ValueError(f'No frames loaded: {path}')

    while len(frames) < num_frames:
        frames.append(frames[-1])

    video = np.stack(frames)
    video = torch.tensor(video, dtype=torch.float32)
    video = video.permute(3, 0, 1, 2)

    return video



In [ ]:
row = train_df.iloc[0]
video_path = data_root / row['clip_path']

x_rgb = load_video(video_path)
x_gray = load_video(video_path, grayscale=True)

print(video_path)
print('RGB shape:', x_rgb.shape)
print('grayscale shape:', x_gray.shape)
print('RGB min/max:', x_rgb.min().item(), x_rgb.max().item())
print('grayscale min/max:', x_gray.min().item(), x_gray.max().item())
print('word:', row['word'])
print('class id:', word_to_idx[row['word']])



## 6. PyTorch Dataset and DataLoader

Dataset returns one example: (video_tensor, class_id).

DataLoader groups examples into batches.

In [ ]:
class LipReadingDataset(Dataset):
    def __init__(self, df, data_root, word_to_idx, grayscale=False):
        self.df = df.reset_index(drop=True)
        self.data_root = data_root
        self.word_to_idx = word_to_idx
        self.grayscale = grayscale

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        video_path = self.data_root / row['clip_path']
        x = load_video(video_path, grayscale=self.grayscale)

        word = row['word']
        y = self.word_to_idx[word]
        y = torch.tensor(y, dtype=torch.long)

        return x, y



In [ ]:
train_dataset = LipReadingDataset(train_df, data_root, word_to_idx)
val_dataset = LipReadingDataset(val_df, data_root, word_to_idx)
test_dataset = LipReadingDataset(test_df, data_root, word_to_idx)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

batch_x, batch_y = next(iter(train_loader))
print('batch_x:', batch_x.shape)
print('batch_y:', batch_y.shape)
print(batch_y)

## 7. Simple 3D-CNN Baseline

Conv3d processes video in time, height, and width.

The model outputs logits with shape [batch_size, num_classes].

In [ ]:
class Simple3DCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d((1, 2, 2)),

            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d((2, 2, 2)),

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d((2, 2, 2)),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool3d((1, 1, 1)),
            nn.Flatten(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
model = Simple3DCNN(num_classes=len(vocab)).to(device)

batch_x = batch_x.to(device)
batch_y = batch_y.to(device)

outputs = model(batch_x)
print('outputs:', outputs.shape)
print('labels:', batch_y.shape)

## 8. Train and Evaluation Functions

Training updates model weights. Evaluation only measures quality and does not update weights.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)

            total_loss += loss.item() * batch_x.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)

    return total_loss / total, correct / total

## 9. Baseline Training

This trains the simple 3D-CNN for a few epochs and saves the best checkpoint by validation accuracy into `/kaggle/working/lipreading_sem4`.


In [ ]:
checkpoint_dir = Path('/kaggle/working/lipreading_sem4')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

model = Simple3DCNN(num_classes=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 5
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_checkpoint_path = checkpoint_dir / 'simple3dcnn_best.pt'

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'epoch {epoch + 1}/{num_epochs} | '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'vocab': vocab,
                'word_to_idx': word_to_idx,
                'idx_to_word': idx_to_word,
                'history': history,
                'best_val_acc': best_val_acc,
            },
            best_checkpoint_path,
        )
        print('saved best:', best_checkpoint_path)

## 10. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='train')
plt.plot(history['val_loss'], label='val')
plt.title('Loss')
plt.xlabel('epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='train')
plt.plot(history['val_acc'], label='val')
plt.title('Accuracy')
plt.xlabel('epoch')
plt.legend()

plt.tight_layout()
plt.show()

## 11. Prediction Diagnostics

These functions check whether the model predicts many classes or collapses into one frequent class.

They show:

- prediction distribution;
- true label distribution;
- per-class accuracy;
- confusion matrix.

In [ ]:
def collect_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(batch_y.cpu().numpy().tolist())

    return all_preds, all_labels


def classification_metrics(preds, labels, num_classes):
    matrix = confusion_matrix_array(preds, labels, num_classes)
    total = int(matrix.sum())
    accuracy = float(np.trace(matrix) / total) if total else 0.0

    class_rows = []
    for class_id in range(num_classes):
        true_positive = int(matrix[class_id, class_id])
        support = int(matrix[class_id, :].sum())
        predicted_count = int(matrix[:, class_id].sum())
        precision = true_positive / predicted_count if predicted_count else 0.0
        recall = true_positive / support if support else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )
        class_rows.append({
            'class_id': class_id,
            'support': support,
            'predicted_count': predicted_count,
            'correct': true_positive,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        })

    active_rows = [row for row in class_rows if row['support'] > 0]
    macro_f1 = float(np.mean([row['f1'] for row in active_rows])) if active_rows else 0.0
    balanced_accuracy = (
        float(np.mean([row['recall'] for row in active_rows]))
        if active_rows
        else 0.0
    )
    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'balanced_accuracy': balanced_accuracy,
        'class_rows': class_rows,
        'confusion_matrix': matrix,
    }


def prediction_report(name, preds, labels, idx_to_word):
    pred_counts = Counter(preds)
    label_counts = Counter(labels)
    metrics = classification_metrics(preds, labels, len(idx_to_word))

    print(name)
    print('accuracy:', round(metrics['accuracy'], 4))
    print('macro-F1:', round(metrics['macro_f1'], 4))
    print('balanced accuracy:', round(metrics['balanced_accuracy'], 4))

    print('\npredicted words:')
    for idx, count in pred_counts.most_common():
        print(idx_to_word[idx], count)

    print('\ntrue words:')
    for idx, count in label_counts.most_common():
        print(idx_to_word[idx], count)

    rows = []
    for class_row in metrics['class_rows']:
        idx = class_row['class_id']
        rows.append({
            'word': idx_to_word[idx],
            'true_count': class_row['support'],
            'pred_count': class_row['predicted_count'],
            'correct': class_row['correct'],
            'precision': class_row['precision'],
            'recall': class_row['recall'],
            'f1': class_row['f1'],
        })

    report_df = pd.DataFrame(rows)
    report_df.attrs['accuracy'] = metrics['accuracy']
    report_df.attrs['macro_f1'] = metrics['macro_f1']
    report_df.attrs['balanced_accuracy'] = metrics['balanced_accuracy']
    display(report_df)
    return report_df


def confusion_matrix_array(preds, labels, num_classes):
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    for true_idx, pred_idx in zip(labels, preds):
        matrix[true_idx, pred_idx] += 1
    return matrix


def plot_confusion_matrix(preds, labels, idx_to_word, title):
    words = [idx_to_word[i] for i in range(len(idx_to_word))]
    matrix = confusion_matrix_array(preds, labels, len(words))

    plt.figure(figsize=(8, 7))
    plt.imshow(matrix, cmap='Blues')
    plt.title(title)
    plt.xlabel('predicted word')
    plt.ylabel('true word')
    plt.xticks(range(len(words)), words, rotation=45, ha='right')
    plt.yticks(range(len(words)), words)
    plt.colorbar()

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            if value:
                plt.text(j, i, str(value), ha='center', va='center')

    plt.tight_layout()
    plt.show()
    return matrix


## 12. Baseline Diagnostics

This checks the plain Simple3DCNN after training with regular CrossEntropyLoss.

In [ ]:
baseline_val_preds, baseline_val_labels = collect_predictions(model, val_loader, device)

baseline_val_report = prediction_report(
    'Simple3DCNN + CrossEntropyLoss on validation',
    baseline_val_preds,
    baseline_val_labels,
    idx_to_word,
)

baseline_confusion = plot_confusion_matrix(
    baseline_val_preds,
    baseline_val_labels,
    idx_to_word,
    'Simple3DCNN + CrossEntropyLoss: validation confusion matrix',
)


## 13. Weighted Loss Experiment

Class weights increase the loss penalty for rare classes and reduce the penalty for frequent classes.

This changes the loss function, but the training batches are still sampled from the original imbalanced dataset.

In [ ]:
train_counts = Counter(train_df['word'])
print(train_counts)

class_weights = []
for word in vocab:
    count = train_counts[word]
    class_weights.append(1.0 / count)

class_weights = torch.tensor(class_weights, dtype=torch.float32)
class_weights = class_weights / class_weights.mean()
class_weights = class_weights.to(device)

print(class_weights)


In [ ]:
weighted_model = Simple3DCNN(num_classes=len(vocab)).to(device)
weighted_criterion = nn.CrossEntropyLoss(weight=class_weights)
weighted_optimizer = torch.optim.Adam(weighted_model.parameters(), lr=1e-3)

num_epochs = 5
history_weighted = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_checkpoint_path = checkpoint_dir / 'simple3dcnn_weighted_best.pt'

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(weighted_model, train_loader, weighted_criterion, weighted_optimizer, device)
    val_loss, val_acc = evaluate(weighted_model, val_loader, weighted_criterion, device)

    history_weighted['train_loss'].append(train_loss)
    history_weighted['train_acc'].append(train_acc)
    history_weighted['val_loss'].append(val_loss)
    history_weighted['val_acc'].append(val_acc)

    print(
        f'epoch {epoch + 1}/{num_epochs} | '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                'epoch': epoch + 1,
                'model_state_dict': weighted_model.state_dict(),
                'optimizer_state_dict': weighted_optimizer.state_dict(),
                'vocab': vocab,
                'history': history_weighted,
                'best_val_acc': best_val_acc,
                'class_weights': class_weights.detach().cpu(),
            },
            best_checkpoint_path,
        )
        print('saved best:', best_checkpoint_path)


## 14. Weighted Loss Diagnostics

This checks whether weighted loss made the model predict more than one class.

In [ ]:
weighted_val_preds, weighted_val_labels = collect_predictions(weighted_model, val_loader, device)

weighted_val_report = prediction_report(
    'Simple3DCNN + weighted CrossEntropyLoss on validation',
    weighted_val_preds,
    weighted_val_labels,
    idx_to_word,
)

weighted_confusion = plot_confusion_matrix(
    weighted_val_preds,
    weighted_val_labels,
    idx_to_word,
    'Simple3DCNN + weighted CrossEntropyLoss: validation confusion matrix',
)


## 15. Balanced Sampler Experiment

`WeightedRandomSampler` changes how training batches are formed.

Instead of only changing the loss value, it samples clips from rare classes more often, so batches become more balanced.

In [ ]:
from torch.utils.data import WeightedRandomSampler

train_label_ids = [word_to_idx[word] for word in train_df['word']]
label_counts = Counter(train_label_ids)

sample_weights = torch.tensor(
    [1.0 / label_counts[label_id] for label_id in train_label_ids],
    dtype=torch.float32,
)

balanced_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

balanced_train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    sampler=balanced_sampler,
    shuffle=False,
    num_workers=2,
)

batch_x_balanced, batch_y_balanced = next(iter(balanced_train_loader))
print('balanced batch labels:', batch_y_balanced)
print('balanced batch words:')
print([idx_to_word[idx.item()] for idx in batch_y_balanced])


In [ ]:
balanced_model = Simple3DCNN(num_classes=len(vocab)).to(device)
balanced_criterion = nn.CrossEntropyLoss()
balanced_optimizer = torch.optim.Adam(balanced_model.parameters(), lr=1e-3)

num_epochs = 5
history_balanced = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_checkpoint_path = checkpoint_dir / 'simple3dcnn_balanced_sampler_best.pt'

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(balanced_model, balanced_train_loader, balanced_criterion, balanced_optimizer, device)
    val_loss, val_acc = evaluate(balanced_model, val_loader, balanced_criterion, device)

    history_balanced['train_loss'].append(train_loss)
    history_balanced['train_acc'].append(train_acc)
    history_balanced['val_loss'].append(val_loss)
    history_balanced['val_acc'].append(val_acc)

    print(
        f'epoch {epoch + 1}/{num_epochs} | '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                'epoch': epoch + 1,
                'model_state_dict': balanced_model.state_dict(),
                'optimizer_state_dict': balanced_optimizer.state_dict(),
                'vocab': vocab,
                'history': history_balanced,
                'best_val_acc': best_val_acc,
            },
            best_checkpoint_path,
        )
        print('saved best:', best_checkpoint_path)


## 16. Balanced Sampler Diagnostics

This checks whether balanced sampling improves prediction diversity and per-class accuracy.

In [ ]:
balanced_val_preds, balanced_val_labels = collect_predictions(balanced_model, val_loader, device)

balanced_val_report = prediction_report(
    'Simple3DCNN + WeightedRandomSampler on validation',
    balanced_val_preds,
    balanced_val_labels,
    idx_to_word,
)

balanced_confusion = plot_confusion_matrix(
    balanced_val_preds,
    balanced_val_labels,
    idx_to_word,
    'Simple3DCNN + WeightedRandomSampler: validation confusion matrix',
)


## 17. Compare Simple3DCNN Experiments

This table compares the best validation accuracy from the three baseline experiments.

Important: accuracy alone can be misleading, so also check the prediction reports and confusion matrices above.


In [ ]:
summary = pd.DataFrame([
    {
        'experiment': 'Simple3DCNN + CrossEntropyLoss',
        'best_val_acc': max(history['val_acc']),
        'last_val_acc': history['val_acc'][-1],
    },
    {
        'experiment': 'Simple3DCNN + weighted CrossEntropyLoss',
        'best_val_acc': max(history_weighted['val_acc']),
        'last_val_acc': history_weighted['val_acc'][-1],
    },
    {
        'experiment': 'Simple3DCNN + WeightedRandomSampler',
        'best_val_acc': max(history_balanced['val_acc']),
        'last_val_acc': history_balanced['val_acc'][-1],
    },
])

summary


## 18. Why Add a Temporal Model

The simple 3D-CNN baseline is useful as a first check, but it collapsed to predicting frequent classes.

For lip reading, the order of mouth movements matters. A stronger baseline should therefore have two parts:

1. A frame encoder that extracts visual features from each mouth frame.
2. A temporal encoder that reads the sequence of frame features.

Here we use a small CNN as the frame encoder and a bidirectional GRU as the temporal encoder.



## 19. RGB and Grayscale Loaders

This creates separate DataLoaders for RGB and grayscale experiments.

RGB input shape: `[batch, 3, frames, height, width]`.

Grayscale input shape: `[batch, 1, frames, height, width]`.



In [ ]:
from torch.utils.data import WeightedRandomSampler


def make_loaders(grayscale=False, batch_size=8, balanced=False, seed=SEED):
    train_dataset = LipReadingDataset(train_df, data_root, word_to_idx, grayscale=grayscale)
    val_dataset = LipReadingDataset(val_df, data_root, word_to_idx, grayscale=grayscale)
    test_dataset = LipReadingDataset(test_df, data_root, word_to_idx, grayscale=grayscale)
    generator = torch.Generator().manual_seed(seed)

    if balanced:
        class_counts = Counter(train_df['word'])
        sample_weights = [1.0 / class_counts[word] for word in train_df['word']]
        sampler = WeightedRandomSampler(
            weights=torch.tensor(sample_weights, dtype=torch.float32),
            num_samples=len(sample_weights),
            replacement=True,
            generator=generator,
        )
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            sampler=sampler,
            num_workers=2,
        )
    else:
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=2,
            generator=generator,
        )

    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, val_loader, test_loader


rgb_train_loader, rgb_val_loader, rgb_test_loader = make_loaders(
    grayscale=False,
    batch_size=8,
)
gray_train_loader, gray_val_loader, gray_test_loader = make_loaders(
    grayscale=True,
    batch_size=8,
)

gray_batch_x, gray_batch_y = next(iter(gray_train_loader))
print('gray batch_x:', gray_batch_x.shape)
print('gray batch_y:', gray_batch_y.shape)
print(gray_batch_y)


## 20. Frame CNN + BiGRU Model

This model processes a clip as a sequence.

Input:

```text
[B, C, T, H, W]
```

Steps:

```text
video
-> split into frames
-> CNN extracts one feature vector per frame
-> BiGRU reads the feature sequence in both time directions
-> Linear classifier predicts the word
```



In [ ]:
class FrameCNNBiGRU(nn.Module):
    def __init__(self, num_classes, in_channels=3, cnn_dim=64, gru_hidden=64, dropout=0.3):
        super().__init__()

        self.frame_encoder = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, cnn_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(cnn_dim),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )

        self.temporal_encoder = nn.GRU(
            input_size=cnn_dim,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(gru_hidden * 2, num_classes),
        )

    def forward(self, x):
        # x: [B, C, T, H, W]
        batch_size, channels, frames, height, width = x.shape

        # Move time next to batch: [B, T, C, H, W]
        x = x.permute(0, 2, 1, 3, 4)

        # Encode every frame with the same CNN: [B*T, C, H, W] -> [B*T, cnn_dim]
        x = x.reshape(batch_size * frames, channels, height, width)
        frame_features = self.frame_encoder(x)

        # Restore the time axis: [B, T, cnn_dim]
        frame_features = frame_features.reshape(batch_size, frames, -1)

        _, hidden = self.temporal_encoder(frame_features)

        # For a bidirectional GRU, the last two hidden states are forward and backward.
        forward_last = hidden[-2]
        backward_last = hidden[-1]
        clip_features = torch.cat([forward_last, backward_last], dim=1)

        logits = self.classifier(clip_features)
        return logits


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


rgb_bigru_model = FrameCNNBiGRU(num_classes=len(vocab), in_channels=3).to(device)
gray_bigru_model = FrameCNNBiGRU(num_classes=len(vocab), in_channels=1).to(device)

print('RGB CNN+BiGRU parameters:', count_parameters(rgb_bigru_model))
print('grayscale CNN+BiGRU parameters:', count_parameters(gray_bigru_model))

with torch.no_grad():
    test_logits = gray_bigru_model(gray_batch_x.to(device))

print('test logits:', test_logits.shape)



## 21. Reusable Controlled Training Helper

Future full-dataset experiments use one protocol:

- speaker-based validation;
- checkpoint selection by validation macro-F1;
- accuracy, macro-F1, balanced accuracy, and per-class recall;
- early stopping after seven epochs without macro-F1 improvement;
- training history saved next to the checkpoint.

The test split is not used here. It is reserved for the final selected model.


In [ ]:
import time


def evaluate_model_detailed(model, loader, criterion, device, num_classes):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)

            total_loss += loss.item() * batch_x.size(0)
            total_examples += batch_y.size(0)
            all_preds.extend(outputs.argmax(dim=1).cpu().tolist())
            all_labels.extend(batch_y.cpu().tolist())

    metrics = classification_metrics(all_preds, all_labels, num_classes)
    metrics['loss'] = total_loss / total_examples
    metrics['preds'] = all_preds
    metrics['labels'] = all_labels
    return metrics


def fit_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs,
    checkpoint_path,
    experiment_name,
    patience=7,
):
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'val_macro_f1': [],
        'val_balanced_acc': [],
        'epoch_seconds': [],
    }
    best_val_macro_f1 = -1.0
    best_val_acc = 0.0
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    trainable_parameters = count_parameters(model)

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.perf_counter()
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
        )
        val_metrics = evaluate_model_detailed(
            model,
            val_loader,
            criterion,
            device,
            num_classes=len(vocab),
        )
        epoch_seconds = time.perf_counter() - epoch_start

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_macro_f1'].append(val_metrics['macro_f1'])
        history['val_balanced_acc'].append(val_metrics['balanced_accuracy'])
        history['epoch_seconds'].append(epoch_seconds)

        print(
            f'{experiment_name} | epoch {epoch}/{num_epochs} | '
            f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
            f'val_loss={val_metrics["loss"]:.4f} '
            f'val_acc={val_metrics["accuracy"]:.4f} '
            f'val_macro_f1={val_metrics["macro_f1"]:.4f} '
            f'val_bal_acc={val_metrics["balanced_accuracy"]:.4f} | '
            f'{epoch_seconds:.1f}s'
        )

        improved = (
            val_metrics['macro_f1'] > best_val_macro_f1
            or (
                val_metrics['macro_f1'] == best_val_macro_f1
                and val_metrics['loss'] < best_val_loss
            )
        )
        if improved:
            best_val_macro_f1 = val_metrics['macro_f1']
            best_val_acc = val_metrics['accuracy']
            best_val_loss = val_metrics['loss']
            epochs_without_improvement = 0
            torch.save(
                {
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'vocab': vocab,
                    'word_to_idx': word_to_idx,
                    'idx_to_word': idx_to_word,
                    'history': history,
                    'best_val_macro_f1': best_val_macro_f1,
                    'best_val_acc': best_val_acc,
                    'best_val_loss': best_val_loss,
                    'experiment_name': experiment_name,
                    'trainable_parameters': trainable_parameters,
                },
                checkpoint_path,
            )
            print('saved best by validation macro-F1:', checkpoint_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f'early stop: validation macro-F1 did not improve for {patience} epochs')
            break

    history_df = pd.DataFrame(history)
    history_path = checkpoint_path.with_suffix('.history.csv')
    history_df.to_csv(history_path, index=False)
    print('saved history:', history_path)
    return history, best_val_macro_f1


## 22. Stable Tiny Overfit Test

This is a diagnostic experiment, not a validation score. The model deliberately trains and evaluates on the same 16 or 32 clips.

The previous run reached 87.5% on 16 clips and 65.6% on 32 clips. This version removes the main sources of instability:

- all random seeds are fixed;
- videos are decoded once and cached in memory;
- `num_workers=0`;
- the diagnostic model uses `GroupNorm` instead of batch-dependent `BatchNorm`;
- dropout, augmentation, and weight decay are disabled;
- training may continue for 150 epochs and stops at 100% memorization.

PASS criteria:

```text
16 clips -> at least 95%
32 clips -> at least 90%
```

The block also checks paths, label round trips, duplicate tensors with conflicting labels, and motion between frames.


In [ ]:
import hashlib
import random


def set_reproducible_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def make_round_robin_tiny_df(df, n_examples=16, random_state=42):
    rng = np.random.default_rng(random_state)
    groups = []

    for word in sorted(df['word'].unique()):
        group = df[df['word'] == word].copy().reset_index(drop=True)
        order = rng.permutation(len(group))
        groups.append(group.iloc[order].reset_index(drop=True))

    rows = []
    position = 0
    while len(rows) < n_examples:
        added = False
        for group in groups:
            if position < len(group):
                rows.append(group.iloc[position])
                added = True
                if len(rows) >= n_examples:
                    break
        if not added:
            break
        position += 1

    return pd.DataFrame(rows).sample(
        frac=1.0,
        random_state=random_state,
    ).reset_index(drop=True)


class CachedTinyDataset(Dataset):
    """Decode every tiny-test clip once, then reuse the same tensors."""

    def __init__(self, df, data_root, word_to_idx, idx_to_word):
        self.df = df.reset_index(drop=True).copy()
        videos = []
        labels = []
        fingerprints = []
        motion_scores = []
        pixel_stds = []
        round_trip_ok = []

        for _, row in self.df.iterrows():
            video_path = data_root / row['clip_path']
            if not video_path.exists():
                raise FileNotFoundError(f'Tiny-overfit clip does not exist: {video_path}')

            video = load_video(video_path, grayscale=True)
            label = word_to_idx[row['word']]

            videos.append(video)
            labels.append(label)
            fingerprints.append(hashlib.sha256(video.numpy().tobytes()).hexdigest())
            motion_scores.append((video[:, 1:] - video[:, :-1]).abs().mean().item())
            pixel_stds.append(video.std().item())
            round_trip_ok.append(idx_to_word[label] == row['word'])

        self.videos = torch.stack(videos)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.df['class_id'] = labels
        self.df['round_trip_ok'] = round_trip_ok
        self.df['tensor_fingerprint'] = fingerprints
        self.df['motion_score'] = motion_scores
        self.df['pixel_std'] = pixel_stds

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self.videos[idx], self.labels[idx]


def audit_cached_tiny_dataset(dataset):
    audit_df = dataset.df.copy()
    conflicting_hashes = (
        audit_df.groupby('tensor_fingerprint')['class_id']
        .nunique()
        .loc[lambda counts: counts > 1]
        .index
        .tolist()
    )
    zero_motion_count = int((audit_df['motion_score'] <= 0.0).sum())
    zero_variance_count = int((audit_df['pixel_std'] <= 0.0).sum())
    round_trip_ok = bool(audit_df['round_trip_ok'].all())

    print('cached tensor shape:', dataset.videos.shape)
    print('label round trips valid:', round_trip_ok)
    print('conflicting duplicate tensors:', len(conflicting_hashes))
    print('zero-motion clips:', zero_motion_count)
    print('zero-variance clips:', zero_variance_count)
    print('motion score min/mean:', audit_df['motion_score'].min(), audit_df['motion_score'].mean())
    display(
        audit_df[
            ['clip_path', 'word', 'speaker_id', 'class_id', 'motion_score', 'pixel_std']
        ]
    )

    data_checks_passed = (
        round_trip_ok
        and not conflicting_hashes
        and zero_motion_count == 0
        and zero_variance_count == 0
    )
    return audit_df, data_checks_passed


class TinyFrameCNNBiGRU(nn.Module):
    """Small deterministic diagnostic model with batch-independent normalization."""

    def __init__(self, num_classes, cnn_dim=64, gru_hidden=64):
        super().__init__()
        self.frame_encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.GroupNorm(4, 16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.GroupNorm(4, 32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, cnn_dim, kernel_size=3, padding=1),
            nn.GroupNorm(8, cnn_dim),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )
        self.temporal_encoder = nn.GRU(
            input_size=cnn_dim,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Linear(gru_hidden * 2, num_classes)

    def forward(self, x):
        batch_size, channels, frames, height, width = x.shape
        x = x.permute(0, 2, 1, 3, 4)
        x = x.reshape(batch_size * frames, channels, height, width)
        frame_features = self.frame_encoder(x)
        frame_features = frame_features.reshape(batch_size, frames, -1)
        _, hidden = self.temporal_encoder(frame_features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


def collect_tiny_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            outputs = model(batch_x.to(device))
            all_preds.extend(outputs.argmax(dim=1).cpu().tolist())
            all_labels.extend(batch_y.tolist())
    return all_preds, all_labels


def show_tiny_error_triplets(dataset, error_df, output_path):
    if error_df.empty:
        print('No remaining tiny-overfit errors.')
        return None

    n_rows = len(error_df)
    fig, axes = plt.subplots(n_rows, 3, figsize=(9, 2.8 * n_rows), squeeze=False)

    for row_number, error in error_df.reset_index(drop=True).iterrows():
        dataset_index = int(error['dataset_index'])
        video = dataset.videos[dataset_index][0]
        frame_ids = [0, video.shape[0] // 2, video.shape[0] - 1]
        titles = ['first frame', 'middle frame', 'last frame']

        for column, (frame_id, title) in enumerate(zip(frame_ids, titles)):
            axes[row_number, column].imshow(video[frame_id].numpy(), cmap='gray', vmin=0, vmax=1)
            axes[row_number, column].axis('off')
            axes[row_number, column].set_title(
                f"{title}\ntrue={error['true_word']} pred={error['pred_word']}"
            )

    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('saved error triplets:', output_path)
    return output_path


def run_stable_tiny_overfit_test(
    n_examples,
    pass_threshold,
    num_epochs=150,
    batch_size=4,
    learning_rate=3e-3,
    seed=42,
):
    set_reproducible_seed(seed)
    tiny_df = make_round_robin_tiny_df(
        train_df,
        n_examples=n_examples,
        random_state=seed + n_examples,
    )
    print('tiny subset size:', len(tiny_df))
    print('word counts:')
    print(tiny_df['word'].value_counts())
    print('speakers:', sorted(tiny_df['speaker_id'].unique()))

    tiny_dataset = CachedTinyDataset(tiny_df, data_root, word_to_idx, idx_to_word)
    audit_df, data_checks_passed = audit_cached_tiny_dataset(tiny_dataset)

    generator = torch.Generator().manual_seed(seed)
    tiny_train_loader = DataLoader(
        tiny_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        generator=generator,
    )
    tiny_eval_loader = DataLoader(
        tiny_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    tiny_model = TinyFrameCNNBiGRU(num_classes=len(vocab)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        tiny_model.parameters(),
        lr=learning_rate,
        weight_decay=0.0,
    )

    history_rows = []
    best_memorization_acc = -1.0
    best_eval_loss = float('inf')
    checkpoint_path = checkpoint_dir / f'stable_tiny_overfit_{n_examples}_best.pt'

    for epoch in range(1, num_epochs + 1):
        train_loss, train_step_acc = train_one_epoch(
            tiny_model,
            tiny_train_loader,
            criterion,
            optimizer,
            device,
        )
        eval_loss, memorization_acc = evaluate(
            tiny_model,
            tiny_eval_loader,
            criterion,
            device,
        )
        history_rows.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_step_acc': train_step_acc,
            'eval_loss_on_same_clips': eval_loss,
            'memorization_acc': memorization_acc,
        })

        improved = (
            memorization_acc > best_memorization_acc
            or (
                memorization_acc == best_memorization_acc
                and eval_loss < best_eval_loss
            )
        )
        if improved:
            best_memorization_acc = memorization_acc
            best_eval_loss = eval_loss
            torch.save(
                {
                    'epoch': epoch,
                    'model_state_dict': tiny_model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'vocab': vocab,
                    'word_to_idx': word_to_idx,
                    'idx_to_word': idx_to_word,
                    'best_memorization_acc': best_memorization_acc,
                    'best_eval_loss': best_eval_loss,
                    'n_examples': n_examples,
                    'seed': seed,
                },
                checkpoint_path,
            )

        if epoch == 1 or epoch % 10 == 0 or memorization_acc >= 1.0:
            print(
                f'tiny {n_examples} | epoch {epoch}/{num_epochs} | '
                f'train_loss={train_loss:.4f} | same-clips loss={eval_loss:.4f} | '
                f'memorization_acc={memorization_acc:.4f} | best={best_memorization_acc:.4f}'
            )

        if memorization_acc >= 1.0:
            print('early stop: perfect memorization reached')
            break

    history_df = pd.DataFrame(history_rows)
    history_path = checkpoint_dir / f'stable_tiny_overfit_{n_examples}_history.csv'
    history_df.to_csv(history_path, index=False)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    tiny_model.load_state_dict(checkpoint['model_state_dict'])
    preds, labels = collect_tiny_predictions(tiny_model, tiny_eval_loader, device)

    error_rows = []
    for dataset_index, (pred, label) in enumerate(zip(preds, labels)):
        if pred == label:
            continue
        metadata = audit_df.iloc[dataset_index]
        error_rows.append({
            'dataset_index': dataset_index,
            'clip_path': metadata['clip_path'],
            'speaker_id': metadata['speaker_id'],
            'true_id': label,
            'pred_id': pred,
            'true_word': idx_to_word[label],
            'pred_word': idx_to_word[pred],
            'motion_score': metadata['motion_score'],
            'pixel_std': metadata['pixel_std'],
        })

    error_columns = [
        'dataset_index', 'clip_path', 'speaker_id', 'true_id', 'pred_id',
        'true_word', 'pred_word', 'motion_score', 'pixel_std',
    ]
    errors_df = pd.DataFrame(error_rows, columns=error_columns)
    errors_path = checkpoint_dir / f'stable_tiny_overfit_{n_examples}_errors.csv'
    errors_df.to_csv(errors_path, index=False)

    triplets_path = checkpoint_dir / f'stable_tiny_overfit_{n_examples}_error_triplets.png'
    show_tiny_error_triplets(tiny_dataset, errors_df, triplets_path)

    passed = data_checks_passed and best_memorization_acc >= pass_threshold
    verdict = (
        f'PASS: best memorization accuracy {best_memorization_acc:.1%} '
        f'reached the {pass_threshold:.0%} threshold.'
        if passed
        else (
            f'CHECK: best memorization accuracy {best_memorization_acc:.1%}; '
            f'required {pass_threshold:.0%}. Inspect the saved errors and data checks.'
        )
    )
    print(verdict)
    print('saved history:', history_path)
    print('saved errors:', errors_path)

    return {
        'n_examples': n_examples,
        'model': tiny_model,
        'history': history_df,
        'best_memorization_acc': best_memorization_acc,
        'pass_threshold': pass_threshold,
        'data_checks_passed': data_checks_passed,
        'passed': passed,
        'checkpoint_path': checkpoint_path,
        'history_path': history_path,
        'errors_path': errors_path,
        'error_triplets_path': triplets_path if not errors_df.empty else None,
        'errors': errors_df,
        'verdict': verdict,
    }

## 23. Stable Tiny Overfit On 16 Clips

This is the first gate. It passes at 95% or higher and stops early at perfect memorization.


In [ ]:
tiny16_result = run_stable_tiny_overfit_test(
    n_examples=16,
    pass_threshold=0.95,
    num_epochs=150,
    batch_size=4,
    learning_rate=3e-3,
    seed=42,
)


## 24. Stable Tiny Overfit On 32 Clips

This is the second gate. It passes at 90% or higher and stops early at perfect memorization.


In [ ]:
tiny32_result = run_stable_tiny_overfit_test(
    n_examples=32,
    pass_threshold=0.90,
    num_epochs=150,
    batch_size=4,
    learning_rate=3e-3,
    seed=42,
)


## 25. Tiny Overfit Decision Gate

Continue to the full-dataset experiments only when both rows show `passed=True`.

- PASS: the data-loading, labels, forward pass, loss, backward pass, and optimizer can learn the training examples.
- CHECK: stop here and inspect the generated error CSV and first/middle/last-frame image.

This test does not measure generalization because it deliberately evaluates the same clips used for training.


In [ ]:
tiny_overfit_summary = pd.DataFrame([
    {
        'n_examples': result['n_examples'],
        'best_memorization_acc': result['best_memorization_acc'],
        'required_acc': result['pass_threshold'],
        'data_checks_passed': result['data_checks_passed'],
        'passed': result['passed'],
        'epochs_ran': len(result['history']),
        'checkpoint_path': str(result['checkpoint_path']),
        'errors_path': str(result['errors_path']),
    }
    for result in [tiny16_result, tiny32_result]
])

display(tiny_overfit_summary)

tiny_pipeline_pass = bool(tiny_overfit_summary['passed'].all())
if tiny_pipeline_pass:
    print('PIPELINE PASS: both tiny-overfit checks passed. Full-dataset experiments may continue.')
else:
    print('PIPELINE CHECK: stop here. Do not run the heavy model cells yet.')


## 26. Train CNN+BiGRU on RGB Clips

This is the stronger temporal baseline with RGB input.



In [ ]:
set_reproducible_seed(SEED)

rgb_bigru_model = FrameCNNBiGRU(num_classes=len(vocab), in_channels=3).to(device)
rgb_bigru_criterion = nn.CrossEntropyLoss()
rgb_bigru_optimizer = torch.optim.AdamW(rgb_bigru_model.parameters(), lr=1e-3, weight_decay=1e-4)
rgb_bigru_checkpoint_path = checkpoint_dir / 'framecnn_bigru_rgb_best.pt'

rgb_bigru_history, rgb_bigru_best_val_macro_f1 = fit_model(
    model=rgb_bigru_model,
    train_loader=rgb_train_loader,
    val_loader=rgb_val_loader,
    criterion=rgb_bigru_criterion,
    optimizer=rgb_bigru_optimizer,
    device=device,
    num_epochs=8,
    checkpoint_path=rgb_bigru_checkpoint_path,
    experiment_name='FrameCNN+BiGRU RGB',
)

rgb_bigru_checkpoint = torch.load(rgb_bigru_checkpoint_path, map_location=device)
rgb_bigru_model.load_state_dict(rgb_bigru_checkpoint['model_state_dict'])
print('loaded best RGB checkpoint:', rgb_bigru_checkpoint_path)
print('best RGB val_acc:', rgb_bigru_checkpoint['best_val_acc'])
print('best RGB val_macro_f1:', rgb_bigru_checkpoint['best_val_macro_f1'])



## 27. RGB CNN+BiGRU Diagnostics

Again, accuracy alone is not enough. We check whether the model predicts several classes or collapses to one word.



In [ ]:
rgb_bigru_val_preds, rgb_bigru_val_labels = collect_predictions(rgb_bigru_model, rgb_val_loader, device)

rgb_bigru_report = prediction_report(
    'FrameCNN+BiGRU RGB on validation',
    rgb_bigru_val_preds,
    rgb_bigru_val_labels,
    idx_to_word,
)

plot_confusion_matrix(
    rgb_bigru_val_preds,
    rgb_bigru_val_labels,
    idx_to_word,
    'FrameCNN+BiGRU RGB: validation confusion matrix',
)



## 28. Train CNN+BiGRU on Grayscale Clips

This checks whether grayscale is a useful optimization. If quality is similar or better than RGB, grayscale is preferable because the model receives less irrelevant color information.



In [ ]:
set_reproducible_seed(SEED)

gray_bigru_model = FrameCNNBiGRU(num_classes=len(vocab), in_channels=1).to(device)
gray_bigru_criterion = nn.CrossEntropyLoss()
gray_bigru_optimizer = torch.optim.AdamW(gray_bigru_model.parameters(), lr=1e-3, weight_decay=1e-4)
gray_bigru_checkpoint_path = checkpoint_dir / 'framecnn_bigru_gray_best.pt'

gray_bigru_history, gray_bigru_best_val_macro_f1 = fit_model(
    model=gray_bigru_model,
    train_loader=gray_train_loader,
    val_loader=gray_val_loader,
    criterion=gray_bigru_criterion,
    optimizer=gray_bigru_optimizer,
    device=device,
    num_epochs=30,
    checkpoint_path=gray_bigru_checkpoint_path,
    experiment_name='FrameCNN+BiGRU grayscale',
)

gray_bigru_checkpoint = torch.load(gray_bigru_checkpoint_path, map_location=device)
gray_bigru_model.load_state_dict(gray_bigru_checkpoint['model_state_dict'])
print('loaded best grayscale checkpoint:', gray_bigru_checkpoint_path)
print('best grayscale val_acc:', gray_bigru_checkpoint['best_val_acc'])
print('best grayscale val_macro_f1:', gray_bigru_checkpoint['best_val_macro_f1'])



## 29. Grayscale CNN+BiGRU Diagnostics

This shows whether grayscale reduces collapse to frequent words or loses useful information.



In [ ]:
gray_bigru_val_preds, gray_bigru_val_labels = collect_predictions(gray_bigru_model, gray_val_loader, device)

gray_bigru_report = prediction_report(
    'FrameCNN+BiGRU grayscale on validation',
    gray_bigru_val_preds,
    gray_bigru_val_labels,
    idx_to_word,
)

plot_confusion_matrix(
    gray_bigru_val_preds,
    gray_bigru_val_labels,
    idx_to_word,
    'FrameCNN+BiGRU grayscale: validation confusion matrix',
)



## 30. Compare All Experiments

This table is the main checkpoint for deciding what to do next: improve the model more, clean crops, or expand the dataset.



In [ ]:
all_experiments_summary = pd.DataFrame([
    {
        'experiment': 'Simple3DCNN + CrossEntropyLoss',
        'best_val_acc': max(history['val_acc']),
        'last_val_acc': history['val_acc'][-1],
    },
    {
        'experiment': 'Simple3DCNN + weighted CrossEntropyLoss',
        'best_val_acc': max(history_weighted['val_acc']),
        'last_val_acc': history_weighted['val_acc'][-1],
    },
    {
        'experiment': 'Simple3DCNN + WeightedRandomSampler',
        'best_val_acc': max(history_balanced['val_acc']),
        'last_val_acc': history_balanced['val_acc'][-1],
    },
    {
        'experiment': 'FrameCNN+BiGRU RGB',
        'best_val_acc': max(rgb_bigru_history['val_acc']),
        'last_val_acc': rgb_bigru_history['val_acc'][-1],
    },
    {
        'experiment': 'FrameCNN+BiGRU grayscale',
        'best_val_acc': max(gray_bigru_history['val_acc']),
        'last_val_acc': gray_bigru_history['val_acc'][-1],
    },
])

all_experiments_summary



## 31. Random Split Sanity Check

The speaker-based split is the honest evaluation setup, because validation and test speakers are unseen during training.

But it is also much harder. This sanity check uses the existing `random_top10` split to answer one question:

```text
Can the model learn the visual word classes at all when speakers are mixed across train/validation/test?
```

If random split works much better than speaker split, the main problem is speaker generalization. If it also fails, the problem is likely dataset quality, alignment/crops, or model capacity.



In [ ]:
def find_split_dir(split_variant):
    candidates = [
        data_root / split_name / split_variant,
        data_root / 'ml_splits_mouth_crops_padded_no_spk09_10' / split_variant,
    ]

    for candidate in candidates:
        if (candidate / 'train.csv').exists():
            return candidate

    raise FileNotFoundError(f'Could not find split: {split_variant}')


random_split_dir = find_split_dir('random_top10')
print('random split:', random_split_dir)

random_train_df = pd.read_csv(random_split_dir / 'train.csv')
random_val_df = pd.read_csv(random_split_dir / 'val.csv')
random_test_df = pd.read_csv(random_split_dir / 'test.csv')

random_vocab_path = random_split_dir / 'vocab.txt'
if random_vocab_path.exists():
    random_vocab = [line.strip() for line in random_vocab_path.read_text(encoding='utf-8').splitlines() if line.strip()]
else:
    random_vocab = sorted(random_train_df['word'].unique())

random_word_to_idx = {word: idx for idx, word in enumerate(random_vocab)}
random_idx_to_word = {idx: word for word, idx in random_word_to_idx.items()}

print('random train:', len(random_train_df))
print('random val:', len(random_val_df))
print('random test:', len(random_test_df))
print('random vocab:', random_vocab)
print('random train speakers:', sorted(random_train_df['speaker_id'].unique()))
print('random val speakers:', sorted(random_val_df['speaker_id'].unique()))
print('random test speakers:', sorted(random_test_df['speaker_id'].unique()))

print('\nRandom train word counts:')
print(random_train_df['word'].value_counts())
print('\nRandom validation word counts:')
print(random_val_df['word'].value_counts())




## 32. Train Grayscale CNN+BiGRU on Random Split

We use grayscale here because it was at least as good as RGB on the speaker-based split and is cheaper to train.



In [ ]:
random_gray_train_dataset = LipReadingDataset(random_train_df, data_root, random_word_to_idx, grayscale=True)
random_gray_val_dataset = LipReadingDataset(random_val_df, data_root, random_word_to_idx, grayscale=True)
random_gray_test_dataset = LipReadingDataset(random_test_df, data_root, random_word_to_idx, grayscale=True)

random_loader_generator = torch.Generator().manual_seed(SEED)

random_gray_train_loader = DataLoader(random_gray_train_dataset, batch_size=8, shuffle=True, num_workers=2, generator=random_loader_generator)
random_gray_val_loader = DataLoader(random_gray_val_dataset, batch_size=8, shuffle=False, num_workers=2)
random_gray_test_loader = DataLoader(random_gray_test_dataset, batch_size=8, shuffle=False, num_workers=2)

set_reproducible_seed(SEED)

random_gray_model = FrameCNNBiGRU(num_classes=len(random_vocab), in_channels=1).to(device)
random_gray_criterion = nn.CrossEntropyLoss()
random_gray_optimizer = torch.optim.AdamW(random_gray_model.parameters(), lr=1e-3, weight_decay=1e-4)
random_gray_checkpoint_path = checkpoint_dir / 'framecnn_bigru_gray_random_split_best.pt'

random_gray_history, random_gray_best_val_macro_f1 = fit_model(
    model=random_gray_model,
    train_loader=random_gray_train_loader,
    val_loader=random_gray_val_loader,
    criterion=random_gray_criterion,
    optimizer=random_gray_optimizer,
    device=device,
    num_epochs=30,
    checkpoint_path=random_gray_checkpoint_path,
    experiment_name='FrameCNN+BiGRU grayscale random split',
)

random_gray_checkpoint = torch.load(random_gray_checkpoint_path, map_location=device)
random_gray_model.load_state_dict(random_gray_checkpoint['model_state_dict'])
print('loaded best random split checkpoint:', random_gray_checkpoint_path)
print('best random split val_acc:', random_gray_checkpoint['best_val_acc'])
print('best random split val_macro_f1:', random_gray_checkpoint['best_val_macro_f1'])



## 33. Random Split Diagnostics

If this block looks much better than the speaker-based diagnostics, the model is learning speaker-specific visual patterns and does not yet generalize well to unseen speakers.



In [ ]:
random_gray_val_preds, random_gray_val_labels = collect_predictions(random_gray_model, random_gray_val_loader, device)

random_gray_report = prediction_report(
    'FrameCNN+BiGRU grayscale on random validation split',
    random_gray_val_preds,
    random_gray_val_labels,
    random_idx_to_word,
)

plot_confusion_matrix(
    random_gray_val_preds,
    random_gray_val_labels,
    random_idx_to_word,
    'FrameCNN+BiGRU grayscale random split: validation confusion matrix',
)



## 34. Final Comparison With Sanity Check

Use the speaker-based rows for honest quality. Use the random-split row only as a diagnostic sanity check.



In [ ]:
final_experiments_summary = pd.DataFrame([
    {
        'experiment': 'Simple3DCNN + CrossEntropyLoss, speaker split',
        'best_val_acc': max(history['val_acc']),
        'last_val_acc': history['val_acc'][-1],
        'evaluation_type': 'speaker-based',
    },
    {
        'experiment': 'Simple3DCNN + weighted CrossEntropyLoss, speaker split',
        'best_val_acc': max(history_weighted['val_acc']),
        'last_val_acc': history_weighted['val_acc'][-1],
        'evaluation_type': 'speaker-based',
    },
    {
        'experiment': 'Simple3DCNN + WeightedRandomSampler, speaker split',
        'best_val_acc': max(history_balanced['val_acc']),
        'last_val_acc': history_balanced['val_acc'][-1],
        'evaluation_type': 'speaker-based',
    },
    {
        'experiment': 'FrameCNN+BiGRU RGB, speaker split',
        'best_val_acc': max(rgb_bigru_history['val_acc']),
        'last_val_acc': rgb_bigru_history['val_acc'][-1],
        'evaluation_type': 'speaker-based',
    },
    {
        'experiment': 'FrameCNN+BiGRU grayscale, speaker split',
        'best_val_acc': max(gray_bigru_history['val_acc']),
        'last_val_acc': gray_bigru_history['val_acc'][-1],
        'evaluation_type': 'speaker-based',
    },
    {
        'experiment': 'FrameCNN+BiGRU grayscale, random split',
        'best_val_acc': max(random_gray_history['val_acc']),
        'last_val_acc': random_gray_history['val_acc'][-1],
        'evaluation_type': 'random sanity check',
    },
])

final_experiments_summary



## 35. Model-Based Clip Audit

The random split sanity check is also weak, so the next step is not to manually review the whole dataset.

Instead, this block scores clips with the trained grayscale CNN+BiGRU model and saves suspicious examples:

```text
high training loss -> possible bad crop/alignment/noisy label
wrong validation prediction -> hard or suspicious validation examples
```

This is only a triage tool. A high-loss clip is not automatically bad, but it is a good candidate for quick visual inspection.



In [ ]:
def score_dataframe(model, df, data_root, word_to_idx, idx_to_word, grayscale=True, batch_size=8, split_label='split'):
    dataset = LipReadingDataset(df, data_root, word_to_idx, grayscale=grayscale)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    criterion_none = nn.CrossEntropyLoss(reduction='none')

    model.eval()
    rows = []
    offset = 0

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            losses = criterion_none(outputs, batch_y)
            probs = torch.softmax(outputs, dim=1)
            pred_conf, preds = probs.max(dim=1)

            batch_size_actual = batch_y.size(0)
            batch_rows = df.iloc[offset:offset + batch_size_actual].copy().reset_index(drop=True)

            for local_idx, row in batch_rows.iterrows():
                true_idx = int(batch_y[local_idx].detach().cpu())
                pred_idx = int(preds[local_idx].detach().cpu())
                rows.append({
                    'split': split_label,
                    'row_index': offset + local_idx,
                    'clip_id': row.get('clip_id', ''),
                    'clip_path': row['clip_path'],
                    'word': row['word'],
                    'speaker_id': row.get('speaker_id', ''),
                    'source': row.get('source', ''),
                    'true_idx': true_idx,
                    'pred_idx': pred_idx,
                    'pred_word': idx_to_word[pred_idx],
                    'loss': float(losses[local_idx].detach().cpu()),
                    'pred_confidence': float(pred_conf[local_idx].detach().cpu()),
                    'correct': bool(pred_idx == true_idx),
                })

            offset += batch_size_actual

    return pd.DataFrame(rows)


def save_audit_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print('saved:', path)
    return path



## 36. Create Contact Sheets For Suspicious Clips

The CSV files are useful, but a contact sheet is faster for human inspection: one image with middle frames from suspicious clips.



In [ ]:
def read_middle_frame(path, size=96):
    cap = cv2.VideoCapture(str(path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return None

    cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)
    ok, frame = cap.read()
    cap.release()

    if not ok:
        return None

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame = cv2.resize(frame, (size, size))
    return frame


def make_contact_sheet(audit_df, data_root, output_path, max_items=36, cols=6, size=96, title='Suspicious clips'):
    items = audit_df.head(max_items).reset_index(drop=True)
    rows = int(np.ceil(len(items) / cols))

    if len(items) == 0:
        print('no items for contact sheet:', output_path)
        return None

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.5))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis('off')

    for i, row in items.iterrows():
        ax = axes[i]
        frame = read_middle_frame(data_root / row['clip_path'], size=size)

        if frame is None:
            ax.text(0.5, 0.5, 'read error', ha='center', va='center')
        else:
            ax.imshow(frame)

        label = f"#{i + 1} loss={row['loss']:.2f}\ntrue={row['word']}\npred={row['pred_word']}\n{row['speaker_id']}"
        ax.set_title(label, fontsize=8)
        ax.axis('off')

    fig.suptitle(title)
    plt.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=160)
    plt.show()
    print('saved:', output_path)
    return output_path




## 37. Run Speaker-Split Audit

This uses the best grayscale CNN+BiGRU model from the speaker-based split.

Outputs:

- `speaker_train_high_loss.csv`: suspicious training clips;
- `speaker_val_wrong_predictions.csv`: validation clips predicted incorrectly;
- two contact-sheet images for quick visual review.



In [ ]:
audit_dir = checkpoint_dir / 'audit'
audit_dir.mkdir(parents=True, exist_ok=True)

speaker_train_scores = score_dataframe(
    gray_bigru_model,
    train_df,
    data_root,
    word_to_idx,
    idx_to_word,
    grayscale=True,
    split_label='speaker_train',
)

speaker_val_scores = score_dataframe(
    gray_bigru_model,
    val_df,
    data_root,
    word_to_idx,
    idx_to_word,
    grayscale=True,
    split_label='speaker_val',
)

speaker_train_high_loss = speaker_train_scores.sort_values('loss', ascending=False).head(100)
speaker_val_wrong = speaker_val_scores[speaker_val_scores['correct'] == False].sort_values('loss', ascending=False).head(100)

save_audit_csv(speaker_train_scores, audit_dir / 'speaker_train_all_scores.csv')
save_audit_csv(speaker_val_scores, audit_dir / 'speaker_val_all_scores.csv')
save_audit_csv(speaker_train_high_loss, audit_dir / 'speaker_train_high_loss_top100.csv')
save_audit_csv(speaker_val_wrong, audit_dir / 'speaker_val_wrong_predictions_top100.csv')

print('speaker train accuracy:', speaker_train_scores['correct'].mean())
print('speaker val accuracy:', speaker_val_scores['correct'].mean())
print('top high-loss training clips:')
display(speaker_train_high_loss.head(20))
print('top wrong validation clips:')
display(speaker_val_wrong.head(20))



## 38. Contact Sheets For Speaker-Split Audit

Open these images first. If many examples show wrong crops, no face, or word boundary errors, clean the dataset before trying a more complex model.



In [ ]:
make_contact_sheet(
    speaker_train_high_loss,
    data_root,
    audit_dir / 'speaker_train_high_loss_top36.jpg',
    max_items=36,
    title='Speaker split: high-loss training clips',
)

make_contact_sheet(
    speaker_val_wrong,
    data_root,
    audit_dir / 'speaker_val_wrong_predictions_top36.jpg',
    max_items=36,
    title='Speaker split: wrong validation predictions',
)



## 39. Optional Random-Split Audit

Run this only if the random-split model has already been trained. It checks whether suspicious clips also appear when speakers are mixed.



In [ ]:
random_train_scores = score_dataframe(
    random_gray_model,
    random_train_df,
    data_root,
    random_word_to_idx,
    random_idx_to_word,
    grayscale=True,
    split_label='random_train',
)

random_val_scores = score_dataframe(
    random_gray_model,
    random_val_df,
    data_root,
    random_word_to_idx,
    random_idx_to_word,
    grayscale=True,
    split_label='random_val',
)

random_train_high_loss = random_train_scores.sort_values('loss', ascending=False).head(100)
random_val_wrong = random_val_scores[random_val_scores['correct'] == False].sort_values('loss', ascending=False).head(100)

save_audit_csv(random_train_scores, audit_dir / 'random_train_all_scores.csv')
save_audit_csv(random_val_scores, audit_dir / 'random_val_all_scores.csv')
save_audit_csv(random_train_high_loss, audit_dir / 'random_train_high_loss_top100.csv')
save_audit_csv(random_val_wrong, audit_dir / 'random_val_wrong_predictions_top100.csv')

make_contact_sheet(
    random_train_high_loss,
    data_root,
    audit_dir / 'random_train_high_loss_top36.jpg',
    max_items=36,
    title='Random split: high-loss training clips',
)

make_contact_sheet(
    random_val_wrong,
    data_root,
    audit_dir / 'random_val_wrong_predictions_top36.jpg',
    max_items=36,
    title='Random split: wrong validation predictions',
)

print('random train accuracy:', random_train_scores['correct'].mean())
print('random val accuracy:', random_val_scores['correct'].mean())



## 40. LRW-Style ResNet18 + BiGRU Baseline

Many word-level lip-reading systems use a stronger visual frontend than a small custom CNN. A common direction is:

```text
mouth ROI frames
-> ResNet-style frame encoder
-> temporal backend, for example BiGRU/TCN
-> word classifier
```

This block implements a reproducible LRW-style baseline with `torchvision.models.resnet18` as the frame encoder and a BiGRU as the temporal encoder.

It is not LRW-pretrained yet. It is the next controlled step after `FrameCNN+BiGRU`.



In [ ]:
try:
    import torchvision.models as tv_models
except ImportError as exc:
    raise ImportError('torchvision is required for the ResNet18 baseline') from exc


def build_resnet18_frame_encoder(in_channels=1, use_imagenet_pretrained=False):
    if use_imagenet_pretrained:
        try:
            weights = tv_models.ResNet18_Weights.DEFAULT
            encoder = tv_models.resnet18(weights=weights)
        except AttributeError:
            encoder = tv_models.resnet18(pretrained=True)
    else:
        try:
            encoder = tv_models.resnet18(weights=None)
        except TypeError:
            encoder = tv_models.resnet18(pretrained=False)

    if in_channels != 3:
        old_conv = encoder.conv1
        new_conv = nn.Conv2d(
            in_channels,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )

        if use_imagenet_pretrained and in_channels == 1:
            with torch.no_grad():
                new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))

        encoder.conv1 = new_conv

    feature_dim = encoder.fc.in_features
    encoder.fc = nn.Identity()
    return encoder, feature_dim


class ResNet18BiGRU(nn.Module):
    def __init__(self, num_classes, in_channels=1, gru_hidden=128, dropout=0.4, use_imagenet_pretrained=False):
        super().__init__()

        self.frame_encoder, feature_dim = build_resnet18_frame_encoder(
            in_channels=in_channels,
            use_imagenet_pretrained=use_imagenet_pretrained,
        )

        self.temporal_encoder = nn.GRU(
            input_size=feature_dim,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(gru_hidden * 2, num_classes),
        )

    def forward(self, x):
        # x: [B, C, T, H, W]
        batch_size, channels, frames, height, width = x.shape
        x = x.permute(0, 2, 1, 3, 4)
        x = x.reshape(batch_size * frames, channels, height, width)

        frame_features = self.frame_encoder(x)
        frame_features = frame_features.reshape(batch_size, frames, -1)

        _, hidden = self.temporal_encoder(frame_features)
        forward_last = hidden[-2]
        backward_last = hidden[-1]
        clip_features = torch.cat([forward_last, backward_last], dim=1)

        logits = self.classifier(clip_features)
        return logits


# Default: no internet/download required. Set True later only if you deliberately want ImageNet initialization.
use_imagenet_pretrained = False

resnet_bigru_model = ResNet18BiGRU(
    num_classes=len(vocab),
    in_channels=1,
    use_imagenet_pretrained=use_imagenet_pretrained,
).to(device)

print('ResNet18+BiGRU parameters:', count_parameters(resnet_bigru_model))

with torch.no_grad():
    test_logits = resnet_bigru_model(gray_batch_x.to(device))

print('test logits:', test_logits.shape)




## 41. Train ResNet18 + BiGRU on Grayscale Speaker Split

This is the first LRW-style experiment on the honest speaker-based split.

It is heavier than the previous small CNN+BiGRU, so the batch size is kept small.



In [ ]:
resnet_gray_train_loader, resnet_gray_val_loader, resnet_gray_test_loader = make_loaders(
    grayscale=True,
    batch_size=4,
)

set_reproducible_seed(SEED)

resnet_bigru_model = ResNet18BiGRU(
    num_classes=len(vocab),
    in_channels=1,
    use_imagenet_pretrained=False,
).to(device)

resnet_bigru_criterion = nn.CrossEntropyLoss()
resnet_bigru_optimizer = torch.optim.AdamW(
    resnet_bigru_model.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)
resnet_bigru_checkpoint_path = checkpoint_dir / 'resnet18_bigru_gray_speaker_best.pt'

resnet_bigru_history, resnet_bigru_best_val_macro_f1 = fit_model(
    model=resnet_bigru_model,
    train_loader=resnet_gray_train_loader,
    val_loader=resnet_gray_val_loader,
    criterion=resnet_bigru_criterion,
    optimizer=resnet_bigru_optimizer,
    device=device,
    num_epochs=5,
    checkpoint_path=resnet_bigru_checkpoint_path,
    experiment_name='ResNet18+BiGRU grayscale speaker split',
)

resnet_bigru_checkpoint = torch.load(resnet_bigru_checkpoint_path, map_location=device)
resnet_bigru_model.load_state_dict(resnet_bigru_checkpoint['model_state_dict'])
print('loaded best ResNet18+BiGRU checkpoint:', resnet_bigru_checkpoint_path)
print('best ResNet18+BiGRU val_acc:', resnet_bigru_checkpoint['best_val_acc'])
print('best ResNet18+BiGRU val_macro_f1:', resnet_bigru_checkpoint['best_val_macro_f1'])



## 42. ResNet18 + BiGRU Diagnostics

As before, check whether the stronger model predicts multiple words or still collapses to frequent classes.



In [ ]:
resnet_bigru_val_preds, resnet_bigru_val_labels = collect_predictions(
    resnet_bigru_model,
    resnet_gray_val_loader,
    device,
)

resnet_bigru_report = prediction_report(
    'ResNet18+BiGRU grayscale on speaker validation split',
    resnet_bigru_val_preds,
    resnet_bigru_val_labels,
    idx_to_word,
)

plot_confusion_matrix(
    resnet_bigru_val_preds,
    resnet_bigru_val_labels,
    idx_to_word,
    'ResNet18+BiGRU grayscale: speaker validation confusion matrix',
)



## 43. Compare With ResNet18 + BiGRU

The ResNet18+BiGRU row should be compared mainly with `FrameCNN+BiGRU grayscale` on the same speaker-based split.



In [ ]:
resnet_experiments_summary = pd.DataFrame([
    {
        'experiment': 'FrameCNN+BiGRU grayscale, speaker split',
        'best_val_acc': max(gray_bigru_history['val_acc']),
        'last_val_acc': gray_bigru_history['val_acc'][-1],
        'model_size': 'small',
    },
    {
        'experiment': 'FrameCNN+BiGRU grayscale, random split',
        'best_val_acc': max(random_gray_history['val_acc']),
        'last_val_acc': random_gray_history['val_acc'][-1],
        'model_size': 'small',
    },
    {
        'experiment': 'ResNet18+BiGRU grayscale, speaker split',
        'best_val_acc': max(resnet_bigru_history['val_acc']),
        'last_val_acc': resnet_bigru_history['val_acc'][-1],
        'model_size': 'larger LRW-style',
    },
])

resnet_experiments_summary



## 44. Frozen Pretrained ResNet18 + BiGRU

The previous ResNet18+BiGRU model trained ResNet18 from scratch on a small dataset. That is difficult because ResNet18 has about 11.7M trainable parameters.

This experiment uses an ImageNet-pretrained ResNet18 frame encoder and freezes it first. Only the BiGRU temporal encoder and word classifier are trained.

This is not lip-reading pretraining yet, but it checks whether general visual features help more than training the frame encoder from scratch.



In [ ]:
def set_requires_grad(module, requires_grad):
    for parameter in module.parameters():
        parameter.requires_grad = requires_grad


frozen_resnet_bigru_model = ResNet18BiGRU(
    num_classes=len(vocab),
    in_channels=1,
    use_imagenet_pretrained=True,
).to(device)

set_requires_grad(frozen_resnet_bigru_model.frame_encoder, False)

trainable_params = count_parameters(frozen_resnet_bigru_model)
total_params = sum(p.numel() for p in frozen_resnet_bigru_model.parameters())

print('frozen pretrained ResNet18+BiGRU total parameters:', total_params)
print('frozen pretrained ResNet18+BiGRU trainable parameters:', trainable_params)

with torch.no_grad():
    test_logits = frozen_resnet_bigru_model(gray_batch_x.to(device))

print('test logits:', test_logits.shape)



## 45. Train Frozen Pretrained ResNet18 + BiGRU

Only the temporal encoder and classifier are updated here. This should be faster and less prone to overfitting than training the whole ResNet18 from scratch.



In [ ]:
frozen_resnet_gray_train_loader, frozen_resnet_gray_val_loader, frozen_resnet_gray_test_loader = make_loaders(
    grayscale=True,
    batch_size=4,
)

set_reproducible_seed(SEED)

frozen_resnet_bigru_model = ResNet18BiGRU(
    num_classes=len(vocab),
    in_channels=1,
    use_imagenet_pretrained=True,
).to(device)
set_requires_grad(frozen_resnet_bigru_model.frame_encoder, False)

frozen_resnet_bigru_criterion = nn.CrossEntropyLoss()
frozen_resnet_bigru_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, frozen_resnet_bigru_model.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
)
frozen_resnet_bigru_checkpoint_path = checkpoint_dir / 'resnet18_bigru_gray_frozen_imagenet_speaker_best.pt'

frozen_resnet_bigru_history, frozen_resnet_bigru_best_val_macro_f1 = fit_model(
    model=frozen_resnet_bigru_model,
    train_loader=frozen_resnet_gray_train_loader,
    val_loader=frozen_resnet_gray_val_loader,
    criterion=frozen_resnet_bigru_criterion,
    optimizer=frozen_resnet_bigru_optimizer,
    device=device,
    num_epochs=30,
    checkpoint_path=frozen_resnet_bigru_checkpoint_path,
    experiment_name='Frozen ImageNet ResNet18+BiGRU grayscale speaker split',
)

frozen_resnet_bigru_checkpoint = torch.load(frozen_resnet_bigru_checkpoint_path, map_location=device)
frozen_resnet_bigru_model.load_state_dict(frozen_resnet_bigru_checkpoint['model_state_dict'])
print('loaded best frozen pretrained checkpoint:', frozen_resnet_bigru_checkpoint_path)
print('best frozen pretrained val_acc:', frozen_resnet_bigru_checkpoint['best_val_acc'])
print('best frozen pretrained val_macro_f1:', frozen_resnet_bigru_checkpoint['best_val_macro_f1'])



## 46. Frozen Pretrained ResNet18 + BiGRU Diagnostics

If this model is better than the randomly initialized ResNet18+BiGRU, pretrained visual features helped.

If it still collapses to frequent words, the bottleneck is likely dataset quality/size, class balance, or the need for domain-specific lip-reading pretraining.



In [ ]:
frozen_resnet_val_preds, frozen_resnet_val_labels = collect_predictions(
    frozen_resnet_bigru_model,
    frozen_resnet_gray_val_loader,
    device,
)

frozen_resnet_report = prediction_report(
    'Frozen ImageNet ResNet18+BiGRU grayscale on speaker validation split',
    frozen_resnet_val_preds,
    frozen_resnet_val_labels,
    idx_to_word,
)

plot_confusion_matrix(
    frozen_resnet_val_preds,
    frozen_resnet_val_labels,
    idx_to_word,
    'Frozen ImageNet ResNet18+BiGRU: speaker validation confusion matrix',
)



## 47. Compare Pretrained Encoder Experiment

This table compares the small model, ResNet18 from scratch, and frozen ImageNet-pretrained ResNet18.



In [ ]:
pretrained_encoder_summary = pd.DataFrame([
    {
        'experiment': 'FrameCNN+BiGRU grayscale, speaker split',
        'best_val_acc': max(gray_bigru_history['val_acc']),
        'last_val_acc': gray_bigru_history['val_acc'][-1],
        'encoder': 'small CNN from scratch',
    },
    {
        'experiment': 'ResNet18+BiGRU grayscale, speaker split',
        'best_val_acc': max(resnet_bigru_history['val_acc']),
        'last_val_acc': resnet_bigru_history['val_acc'][-1],
        'encoder': 'ResNet18 from scratch',
    },
    {
        'experiment': 'Frozen ImageNet ResNet18+BiGRU grayscale, speaker split',
        'best_val_acc': max(frozen_resnet_bigru_history['val_acc']),
        'last_val_acc': frozen_resnet_bigru_history['val_acc'][-1],
        'encoder': 'frozen ImageNet ResNet18',
    },
])

pretrained_encoder_summary



## 48. Next Experiments

Recommended next steps after the frozen pretrained encoder experiment:

1. If frozen ImageNet ResNet18 improves validation accuracy and predicts more classes, fine-tune the last ResNet block.
2. If it does not improve, do not keep scaling architecture blindly.
3. Use the audit CSV/contact sheets to clean only clearly bad clips.
4. Add targeted data for weak classes and more speakers.
5. Later: integrate lip-reading pretrained weights from an LRW repository as a separate task.

